Project Plan: Building the current table from the played games, fitting a Poisson model that
gives each team an attack and a defence rating plus a home advantage; pricing the upcoming
round from it, and simulate the rest of the season for the outright probabilities.

In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import poisson

played   = pd.read_csv("sample_data.csv")
upcoming = pd.read_csv("upcoming_fixtures.csv")
second   = pd.read_csv("second_half_of_season.csv")

In [3]:
# dataset inspection
print("File shapes (rows, cols)")
print("  played   :", played.shape)
print("  upcoming :", upcoming.shape)
print("  second   :", second.shape)

fixtures = pd.concat([played, upcoming, second], ignore_index=True)[["Match ID", "Home ID", "Away ID"]]

teams = pd.unique(fixtures[["Home ID", "Away ID"]].values.ravel())
print("\nTeams:", len(teams))

print("\nFull season checks")
print("  total games      :", len(fixtures))
print("  match IDs        :", fixtures["Match ID"].min(), "to", fixtures["Match ID"].max(),
      "| unique:", fixtures["Match ID"].nunique())
print("  team home games  :", fixtures["Home ID"].value_counts().min(), "to", fixtures["Home ID"].value_counts().max())
print("  team away games  :", fixtures["Away ID"].value_counts().min(), "to", fixtures["Away ID"].value_counts().max())


print("\nSample played games:")
print(played.head().to_string(index=False))

File shapes (rows, cols)
  played   : (160, 7)
  upcoming : (20, 6)
  second   : (200, 3)

Teams: 20

Full season checks
  total games      : 380
  match IDs        : 1 to 380 | unique: 380
  team home games  : 19 to 19
  team away games  : 19 to 19

Sample played games:
 Match ID  Home ID  Away ID    1    X     2 Score
        1       20       15 2.59 3.45  2.79   2:3
        2        9        8 1.31 5.66 10.94   4:0
        3       17        3 6.38 4.53  1.52   1:2
        4        6       11 4.15 3.83  1.88   1:3
        5       18       10 7.05 4.08  1.54   2:2


In [4]:
# Creation of table based on results available

all_fixtures = pd.concat([played, upcoming, second])[["Home ID", "Away ID"]]
assert len(all_fixtures) == 380
assert all_fixtures.drop_duplicates().shape[0] == 380


played[["HG", "AG"]] = played["Score"].str.split(":", expand=True).astype(int)


home_view = played.rename(columns={"Home ID": "Team", "HG": "GF", "AG": "GA"})[["Team", "GF", "GA"]]
away_view = played.rename(columns={"Away ID": "Team", "AG": "GF", "HG": "GA"})[["Team", "GF", "GA"]]
team_games = pd.concat([home_view, away_view], ignore_index=True)


team_games["Pts"] = (team_games.GF > team_games.GA) * 3 + (team_games.GF == team_games.GA) * 1

table = team_games.groupby("Team").agg(
    **{"Matches Played": ("GF", "size"),
       "Goals For":     ("GF", "sum"),
       "Goals Against": ("GA", "sum"),
       "Points":        ("Pts", "sum")}
).reset_index().rename(columns={"Team": "Team ID"})
table["Goal Difference"] = table["Goals For"] - table["Goals Against"]


table = table.sort_values(["Points", "Goal Difference", "Goals For"], ascending=False).reset_index(drop=True)
table.insert(0, "Position", range(1, len(table) + 1))

table = table[["Position", "Team ID", "Matches Played", "Goals For", "Goals Against", "Goal Difference", "Points"]]
table.to_csv("league_table.csv", index=False)
table

,Position,Team ID,Matches Played,Goals For,Goals Against,Goal Difference,Points
0,1,1,16,35,18,17,38
1,2,9,16,39,15,24,37
2,3,12,16,34,12,22,36
3,4,3,16,35,19,16,34
4,5,10,16,22,16,6,27
5,6,7,16,27,22,5,27
6,7,2,16,24,19,5,25
7,8,11,16,32,30,2,25
8,9,4,16,23,27,-4,24
9,10,20,16,32,28,4,23


In [5]:
# Fitting the model - Poisson regression

home_rows = pd.DataFrame({"goals": played["HG"], "team": played["Home ID"], "opponent": played["Away ID"], "home": 1})
away_rows = pd.DataFrame({"goals": played["AG"], "team": played["Away ID"], "opponent": played["Home ID"], "home": 0})
model_df = pd.concat([home_rows, away_rows], ignore_index=True)

poisson_model = smf.glm(
    "goals ~ home + C(team) + C(opponent)",
    data=model_df,
    family=sm.families.Poisson()
).fit()

poisson_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:                  goals   No. Observations:                  320
Model:                            GLM   Df Residuals:                      280
Model Family:                 Poisson   Df Model:                           39
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -451.08
Date:                Wed, 01 Jul 2026   Deviance:                       277.70
Time:                        09:56:23   Pearson chi2:                     226.
No. Iterations:                     5   Pseudo R-squ. (CS):             0.2421
Covariance Type:            nonrobust                                         
=====================================================================================
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.3780      0.300      1.260      0.208      -0.210       0.966
C(team)[T.2]         -0.3159      0.268     -1.181      0.238      -0.840       0.209
C(team)[T.3]          0.0259      0.242      0.107      0.915      -0.448       0.499
C(team)[T.4]         -0.4410      0.270     -1.631      0.103      -0.971       0.089
C(team)[T.5]         -0.7450      0.297     -2.507      0.012      -1.327      -0.163
C(team)[T.6]         -0.2844      0.260     -1.095      0.274      -0.794       0.225
C(team)[T.7]         -0.1859      0.259     -0.718      0.473      -0.694       0.322
C(team)[T.8]         -0.6411      0.298     -2.152      0.031      -1.225      -0.057
C(team)[T.9]          0.1437      0.236      0.609      0.542      -0.319       0.606
C(team)[T.10]        -0.4327      0.274     -1.577      0.115      -0.971       0.105
C(team)[T.11]        -0.0212      0.247     -0.086      0.932      -0.506       0.464
C(team)[T.12]        -0.0010      0.242     -0.004      0.997      -0.474       0.472
C(team)[T.13]        -1.1320      0.347     -3.262      0.001      -1.812      -0.452
C(team)[T.14]        -0.4134      0.275     -1.506      0.132      -0.951       0.125
C(team)[T.15]        -0.2575      0.261     -0.985      0.325      -0.770       0.255
C(team)[T.16]        -0.6926      0.297     -2.334      0.020      -1.274      -0.111
C(team)[T.17]        -0.5636      0.282     -1.995      0.046      -1.117      -0.010
C(team)[T.18]        -0.4567      0.277     -1.646      0.100      -1.001       0.087
C(team)[T.19]        -0.7929      0.311     -2.553      0.011      -1.402      -0.184
C(team)[T.20]        -0.0849      0.247     -0.344      0.731      -0.569       0.399
C(opponent)[T.2]      0.0263      0.331      0.080      0.937      -0.623       0.676
C(opponent)[T.3]      0.0390      0.332      0.118      0.906      -0.611       0.689
C(opponent)[T.4]      0.3439      0.307      1.121      0.262      -0.257       0.945
C(opponent)[T.5]      0.4641      0.300      1.546      0.122      -0.124       1.052
C(opponent)[T.6]      0.4847      0.301      1.611      0.107      -0.105       1.074
C(opponent)[T.7]      0.1449      0.320      0.453      0.651      -0.482       0.772
C(opponent)[T.8]      0.5261      0.299      1.758      0.079      -0.060       1.112
C(opponent)[T.9]     -0.2732      0.351     -0.777      0.437      -0.962       0.416
C(opponent)[T.10]    -0.2157      0.345     -0.624      0.532      -0.893       0.461
C(opponent)[T.11]     0.4437      0.301      1.476      0.140      -0.146       1.033
C(opponent)[T.12]    -0.4172      0.373     -1.118      0.264      -1.149       0.314
C(opponent)[T.13]     0.5735      0.295      1.943      0.052      -0.005       1.152
C(opponent)[T.14]     0.5145      0.297      1.731      0.084      -0.068

In [6]:
# Team Ratings
home_adv = np.exp(poisson_model.params["home"])
print(f"Home advantage factor: {home_adv:.3f}")

params = poisson_model.params
ratings = pd.DataFrame({
    "Team ID": range(1, 21),
    "Attack":  [params.get(f"C(team)[T.{t}]", 0.0) for t in range(1, 21)],
    "Defence": [-params.get(f"C(opponent)[T.{t}]", 0.0) for t in range(1, 21)],
}).round(3)


print(ratings.sort_values("Attack", ascending=False).to_string(index=False))

Home advantage factor: 1.202
 Team ID  Attack  Defence
       9   0.144    0.273
       3   0.026   -0.039
       1   0.000   -0.000
      12  -0.001    0.417
      11  -0.021   -0.444
      20  -0.085   -0.444
       7  -0.186   -0.145
      15  -0.257   -0.309
       6  -0.284   -0.485
       2  -0.316   -0.026
      14  -0.413   -0.515
      10  -0.433    0.216
       4  -0.441   -0.344
      18  -0.457   -0.319
      17  -0.564    0.068
       8  -0.641   -0.526
      16  -0.693   -0.646
       5  -0.745   -0.464
      19  -0.793   -0.407
      13  -1.132   -0.574


In [7]:
# Producing Scorelines
MAX_GOALS = 10  

def expected_goals(home_id, away_id):
    
    rows = pd.DataFrame({"team": [home_id, away_id], "opponent": [away_id, home_id], "home": [1, 0]})
    lam = poisson_model.predict(rows).values
    return lam[0], lam[1]

def match_probs(home_id, away_id):
    lh, la = expected_goals(home_id, away_id)
    
    grid = np.outer(poisson.pmf(np.arange(MAX_GOALS + 1), lh),
                    poisson.pmf(np.arange(MAX_GOALS + 1), la))
    grid /= grid.sum()                  
    home_win = np.tril(grid, -1).sum()  
    draw     = np.trace(grid)           
    away_win = np.triu(grid, 1).sum()   
    return home_win, draw, away_win

In [8]:
# Pricing Matches

probs = np.array([match_probs(r["Home ID"], r["Away ID"]) for _, r in upcoming.iterrows()])
assert np.allclose(probs.sum(axis=1), 1.0)  

upcoming["1"] = probs[:, 0].round(4)
upcoming["X"] = probs[:, 1].round(4)
upcoming["2"] = probs[:, 2].round(4)
upcoming.to_csv("upcoming_fixtures.csv", index=False)
upcoming

,Match ID,Home ID,Away ID,1,X,2
0,161,8,14,0.3637,0.2349,0.4014
1,162,7,13,0.8123,0.1307,0.0570
2,163,19,10,0.1675,0.2721,0.5603
3,164,18,1,0.2053,0.2136,0.5812
4,165,17,4,0.4939,0.2720,0.2341
5,166,20,3,0.2761,0.1972,0.5267
6,167,12,6,0.8087,0.1239,0.0674
7,168,15,11,0.4296,0.2047,0.3657
8,169,9,5,0.8829,0.0842,0.0330
9,170,2,16,0.7463,0.1588,0.0950


In-sample check that shows agreement with the market

In [10]:
# Calibration with the market

played["outcome"] = np.where(played.HG > played.AG, 0, np.where(played.HG == played.AG, 1, 2))

implied = 1.0 / played[["1", "X", "2"]].values
market_probs = implied / implied.sum(axis=1, keepdims=True)
print(f"Average book overround: {implied.sum(axis=1).mean():.3f}  ({implied.sum(axis=1).mean()-1:.1%} margin)")


model_probs = np.array([match_probs(r["Home ID"], r["Away ID"]) for _, r in played.iterrows()])


def log_loss(probs, outcomes):
    picked = probs[np.arange(len(outcomes)), outcomes]
    return -np.log(np.clip(picked, 1e-12, 1)).mean()

oc = played["outcome"].values
print(f"Log-loss   model {log_loss(model_probs, oc):.4f}   market {log_loss(market_probs, oc):.4f}")
print(f"Home-win probability correlation, model vs market: {np.corrcoef(model_probs[:,0], market_probs[:,0])[0,1]:.3f}")

Average book overround: 1.035  (3.5% margin)
Log-loss   model 0.8927   market 0.9130
Home-win probability correlation, model vs market: 0.921


In [11]:
# Setting up the remaining fixtures

base = table.sort_values("Team ID")
pts0 = base["Points"].to_numpy(float)
gd0  = base["Goal Difference"].to_numpy(float)
gf0  = base["Goals For"].to_numpy(float)

remaining = pd.concat([upcoming[["Home ID", "Away ID"]], second[["Home ID", "Away ID"]]], ignore_index=True)


fixture_lams = np.array([expected_goals(h, a) for h, a in zip(remaining["Home ID"], remaining["Away ID"])])
home_idx = remaining["Home ID"].to_numpy() - 1   
away_idx = remaining["Away ID"].to_numpy() - 1

In [12]:
# Simulation
N_SIMS = 50_000
SEED = 42

def simulate_season(n_sims=N_SIMS, seed=SEED):
    rng = np.random.default_rng(seed)
    
    pts = np.tile(pts0, (n_sims, 1))
    gd  = np.tile(gd0,  (n_sims, 1))
    gf  = np.tile(gf0,  (n_sims, 1))

    
    for k in range(len(remaining)):
        hg = rng.poisson(fixture_lams[k, 0], n_sims)
        ag = rng.poisson(fixture_lams[k, 1], n_sims)
        home_win, draw, away_win = hg > ag, hg == ag, hg < ag
        h, a = home_idx[k], away_idx[k]
        pts[:, h] += 3 * home_win + draw
        pts[:, a] += 3 * away_win + draw
        gd[:, h]  += hg - ag
        gd[:, a]  += ag - hg
        gf[:, h]  += hg
        gf[:, a]  += ag

    
    key = pts * 1_000_000 + (gd + 500) * 1_000 + gf
    order = np.argsort(-key, axis=1)   

    win   = np.bincount(order[:, 0],          minlength=20) / n_sims
    top4  = np.bincount(order[:, :4].ravel(),  minlength=20) / n_sims
    releg = np.bincount(order[:, -3:].ravel(), minlength=20) / n_sims
    return win, top4, releg

win, top4, releg = simulate_season()

In [13]:
# Results
assert np.isclose(win.sum(), 1) and np.isclose(top4.sum(), 4) and np.isclose(releg.sum(), 3)

outrights = pd.read_csv("outrights_probabilities.csv")
outrights["Win"]        = win.round(4)
outrights["Top 4"]      = top4.round(4)
outrights["Relegation"] = releg.round(4)
outrights.to_csv("outrights_probabilities.csv", index=False)
outrights.sort_values("Win", ascending=False)

,Team ID,Win,Top 4,Relegation
8,9,0.6115,0.9989,0.0000
11,12,0.2787,0.9941,0.0000
0,1,0.0817,0.9542,0.0000
2,3,0.0279,0.8817,0.0000
6,7,0.0001,0.0636,0.0000
1,2,0.0000,0.0204,0.0000
3,4,0.0000,0.0002,0.0002
4,5,0.0000,0.0000,0.6290
7,8,0.0000,0.0000,0.4920
5,6,0.0000,0.0001,0.0004
